# Laboratorio 08 - Máquinas Vectoriales de Soporte (SVM)
## CC3074 - Mineria de Datos | Universidad del Valle de Guatemala

**Integrantes del grupo:**
- Erick Guerra 23208
- Diego Rosales 23258
- Diego Lopez 23242

**Dataset:** Airbnb Listings (`listings.RData`)

---
## Actividad 1 - Reutilizar los mismos conjuntos de entrenamiento y prueba

En esta actividad se replica el mismo criterio de particion usado en hojas anteriores para garantizar comparabilidad de resultados al probar SVM:
- `test_size=0.2`
- `random_state=42`
- estratificacion por deciles de `log1p(price_num)`

In [1]:
# Imports base para carga, preprocesamiento y split
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadr
from sklearn.model_selection import train_test_split

np.random.seed(42)
print('Imports OK')

Imports OK


In [2]:
# Carga y preprocesamiento consistente con laboratorios anteriores
rdata_path = Path('data/listings.RData')
result = pyreadr.read_r(str(rdata_path))
df = result['listings'].copy()


def parse_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
        errors='coerce',
    )

if 'price' in df.columns:
    df['price_num'] = parse_money(df['price'])

for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df.columns:
        df[f'{col}_num'] = pd.to_numeric(
            df[col].astype(str).str.replace('%', '', regex=False),
            errors='coerce',
        )

candidate_features = [
    'room_type', 'property_type', 'neighbourhood_cleansed',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month', 'review_scores_rating',
    'host_is_superhost', 'host_identity_verified',
    'host_response_rate_num', 'host_acceptance_rate_num',
    'calculated_host_listings_count', 'instant_bookable',
    'latitude', 'longitude', 'price_num',
]

selected_cols = [c for c in candidate_features if c in df.columns]
model_df = df[selected_cols].copy()

model_df = model_df[model_df['price_num'].notna()].copy()
p99_price = model_df['price_num'].quantile(0.99)
model_df = model_df[model_df['price_num'] <= p99_price].copy()

num_cols = [c for c in model_df.select_dtypes(include=np.number).columns if c != 'price_num']
cat_cols = model_df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

for col in num_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in cat_cols:
    model_df[col] = model_df[col].astype(str).replace('nan', 'SinDato')

X = model_df.drop(columns=['price_num'])
y_raw = model_df['price_num']

print(f'Dataset modelado: {model_df.shape[0]:,} filas, {X.shape[1]} features')

Dataset modelado: 75,531 filas, 21 features


In [3]:
# Split identico al usado en laboratorios previos
# Mantiene comparabilidad: mismos parametros y misma logica de estratificacion.
y_log = np.log1p(y_raw)
price_deciles = pd.qcut(y_log, q=10, labels=False, duplicates='drop')

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X,
    y_raw,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=price_deciles,
)

# Variable objetivo binaria para clasificacion (cara vs no cara), igual criterio del lab pasado.
p66 = y_train_raw.quantile(0.66)
y_train = (y_train_raw > p66).astype(int)
y_test = (y_test_raw > p66).astype(int)

print('Split aplicado con los mismos criterios de hojas anteriores:')
print(f'- Train: {X_train.shape[0]:,} filas')
print(f'- Test : {X_test.shape[0]:,} filas')
print(f'- Umbral clase Cara (P66 de train): {p66:.2f}')
print(f'- Proporcion clase 1 en train: {y_train.mean():.3f}')
print(f'- Proporcion clase 1 en test : {y_test.mean():.3f}')

Split aplicado con los mismos criterios de hojas anteriores:
- Train: 60,424 filas
- Test : 15,107 filas
- Umbral clase Cara (P66 de train): 260.00
- Proporcion clase 1 en train: 0.339
- Proporcion clase 1 en test : 0.340


---
## Actividad 2 - Exploracion de datos y transformaciones para SVM

Se exploran los datos de entrenamiento para identificar que transformaciones son necesarias antes de entrenar un modelo de Maquinas Vectoriales de Soporte.

Objetivos:
1. Ver estructura de variables y balance de clases.
2. Detectar aspectos que afectan SVM (escala, categorias, dimensionalidad).
3. Definir un preprocesamiento reproducible para usar en el modelo.

In [4]:
# 2.1 Exploracion general del set de entrenamiento
print('=== Dimensiones ===')
print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test : {y_test.shape}')

num_features = X_train.select_dtypes(include=np.number).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

print('\n=== Tipos de variables ===')
print(f'Numericas : {len(num_features)}')
print(f'Categoricas: {len(cat_features)}')

# Balance de clases para evaluar posibles ajustes de class_weight
class_dist = y_train.value_counts().sort_index()
class_pct = y_train.value_counts(normalize=True).sort_index().mul(100)

print('\n=== Distribucion de clase objetivo (train) ===')
for cls in class_dist.index:
    print(f'Clase {cls}: {class_dist[cls]:,} ({class_pct[cls]:.2f}%)')

=== Dimensiones ===
X_train: (60424, 21)
X_test : (15107, 21)
y_train: (60424,)
y_test : (15107,)

=== Tipos de variables ===
Numericas : 13
Categoricas: 8

=== Distribucion de clase objetivo (train) ===
Clase 0: 39,935 (66.09%)
Clase 1: 20,489 (33.91%)


In [5]:
# 2.2 Exploracion orientada a transformaciones para SVM
# A) Verificacion de valores faltantes (deberia ser 0 tras el preprocesamiento previo)
missing_total = int(X_train.isna().sum().sum())
print(f'Valores faltantes en X_train: {missing_total}')

# B) Cardinalidad de variables categoricas (impacta dimensionalidad por One-Hot)
if len(cat_features) > 0:
    cardinality = pd.Series({c: X_train[c].nunique() for c in cat_features}).sort_values(ascending=False)
    print('\nTop 10 variables categoricas por cardinalidad:')
    print(cardinality.head(10).to_string())

# C) Escala de variables numericas (SVM es sensible a escalas diferentes)
if len(num_features) > 0:
    num_desc = X_train[num_features].describe().T[['mean', 'std', 'min', 'max']]
    print('\nResumen numerico (mean/std/min/max):')
    print(num_desc.head(12).to_string())

    # Ratio de escala aproximado: ayuda a justificar estandarizacion
    scale_ratio = (num_desc['max'] - num_desc['min']) / (num_desc['std'].replace(0, np.nan))
    print('\nVariable con mayor amplitud relativa (max-min)/std:')
    top_scale = scale_ratio.sort_values(ascending=False).head(5)
    print(top_scale.to_string())

Valores faltantes en X_train: 161

Top 10 variables categoricas por cardinalidad:
neighbourhood_cleansed    349
property_type              86
beds                       36
bedrooms                   21
room_type                   4
host_is_superhost           3
host_identity_verified      3
instant_bookable            2

Resumen numerico (mean/std/min/max):
                                      mean         std        min          max
accommodates                      4.834801    2.989583   1.000000    16.000000
bathrooms                         1.618206    0.984900   0.000000    17.000000
minimum_nights                    9.505809   22.646422   1.000000   720.000000
maximum_nights                  468.788296  416.839231   1.000000  3650.000000
availability_365                231.384268  106.548864   0.000000   365.000000
number_of_reviews                55.003972   96.967206   0.000000  1592.000000
reviews_per_month                 1.487919    1.603177   0.010000    61.590000
review_s

In [7]:
# 2.3 Preprocesamiento recomendado para SVM
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features),
    ],
    remainder='drop'
)

# Ajustar SOLO con train para evitar leakage
a = preprocessor_svm.fit_transform(X_train)
b = preprocessor_svm.transform(X_test)

n_train, p_train = a.shape
n_test, p_test = b.shape

print('=== Matriz transformada para SVM ===')
print(f'Train transformado: {n_train:,} filas x {p_train:,} columnas')
print(f'Test transformado : {n_test:,} filas x {p_test:,} columnas')

# Densidad de la matriz (si es dispersa, conviene mantener formato sparse)
if hasattr(a, 'nnz'):
    density = a.nnz / (a.shape[0] * a.shape[1])
    print(f'Densidad matriz train: {density:.6f}')
    print(f'Esparcidad aprox.    : {1 - density:.6f}')
else:
    density = np.count_nonzero(a) / a.size
    print(f'Densidad matriz train: {density:.6f}')
    print(f'Esparcidad aprox.    : {1 - density:.6f}')

=== Matriz transformada para SVM ===
Train transformado: 60,424 filas x 517 columnas
Test transformado : 15,107 filas x 517 columnas
Densidad matriz train: 0.040619
Esparcidad aprox.    : 0.959381


### Explicacion de transformaciones necesarias para SVM

Con base en la exploracion anterior, las transformaciones clave son:

1. Imputar valores faltantes antes de modelar.
   SVM no acepta NaN. Se usa `SimpleImputer(strategy='median')` en numericas y `SimpleImputer(strategy='most_frequent')` en categoricas.

2. Estandarizar variables numericas con `StandardScaler`.
   SVM optimiza margenes usando distancias; si una variable tiene escala mucho mayor que otra, domina el hiperplano y degrada el modelo.

3. Codificar variables categoricas con `OneHotEncoder(handle_unknown='ignore')`.
   SVM requiere entradas numericas. One-Hot evita imponer orden artificial entre categorias y permite manejar categorias nuevas en prueba.

4. Ajustar el preprocesador solo con entrenamiento y luego transformar prueba.
   Esto evita data leakage y garantiza evaluacion valida.

5. Considerar el formato disperso generado por One-Hot.
   La matriz suele quedar muy esparsa; esto influye en la eleccion del kernel y el costo computacional.

6. Mantener el mismo split de actividades previas.
   Asegura comparabilidad justa entre SVM y modelos anteriores.

---
## Actividad 3 - Variable respuesta categorica: barata, media o cara

Para esta actividad se define la variable objetivo multiclase a partir del precio:
- barata
- media
- cara

Los umbrales se calculan solo con el conjunto de entrenamiento para evitar leakage y luego se aplican a train y test.

In [8]:
# 3.1 Construccion de la variable respuesta multiclase
# Regla: usar cuantiles del train para definir barata/media/cara sin leakage.
p33 = y_train_raw.quantile(0.33)
p66 = y_train_raw.quantile(0.66)


def asignar_categoria_precio(price_series: pd.Series, q33: float, q66: float) -> pd.Series:
    return pd.cut(
        price_series,
        bins=[-np.inf, q33, q66, np.inf],
        labels=['barata', 'media', 'cara'],
    )


y_train_cat = asignar_categoria_precio(y_train_raw, p33, p66).astype(str)
y_test_cat = asignar_categoria_precio(y_test_raw, p33, p66).astype(str)

# Version codificada opcional (util para algunos reportes)
cat_to_int = {'barata': 0, 'media': 1, 'cara': 2}
y_train_cat_num = y_train_cat.map(cat_to_int)
y_test_cat_num = y_test_cat.map(cat_to_int)

print('=== Umbrales de precio (calculados en train) ===')
print(f'Q33 (barata/media): {p33:.2f}')
print(f'Q66 (media/cara)  : {p66:.2f}')

print('\n=== Distribucion de clases (train) ===')
print(y_train_cat.value_counts().reindex(['barata', 'media', 'cara']).to_string())

print('\n=== Distribucion de clases (test) ===')
print(y_test_cat.value_counts().reindex(['barata', 'media', 'cara']).to_string())

print('\nMuestras de la variable respuesta categorica (train):')
display(y_train_cat.head())

# Comprobacion de integridad
valid_labels_train = set(y_train_cat.unique()).issubset({'barata', 'media', 'cara'})
valid_labels_test = set(y_test_cat.unique()).issubset({'barata', 'media', 'cara'})
print(f'\nEtiquetas validas en train: {valid_labels_train}')
print(f'Etiquetas validas en test : {valid_labels_test}')

=== Umbrales de precio (calculados en train) ===
Q33 (barata/media): 141.00
Q66 (media/cara)  : 260.00

=== Distribucion de clases (train) ===
price_num
barata    20143
media     19792
cara      20489

=== Distribucion de clases (test) ===
price_num
barata    5042
media     4926
cara      5139

Muestras de la variable respuesta categorica (train):


23421     barata
36797     barata
25953     barata
160615     media
27075     barata
Name: price_num, dtype: str


Etiquetas validas en train: True
Etiquetas validas en test : True
